In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'
import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Machine Learning & Deep Learning libraries
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve
)
from sklearn.feature_selection import mutual_info_classif
import keras
import torch

# Configure global visualization settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

# Strict reproducibility seed configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
keras.utils.set_random_seed(SEED)

print("Environment setup complete. Reproducible random seed set to 42.")
print(f"Keras backend: {keras.backend.backend()} | Torch device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


In [ ]:
# 2.1 Ingest the dataset with fallback path support
import re

csv_paths = [
    '../Excel_Top10_Features.csv',
    'Excel_Top10_Features.csv',
    'D:/UUM/ASSIGNMENT/SEM5/NN/STINK3014_A261_Assignment01/compPersonal/excel/Excel_Top10_Features.csv'
]

df = None
loaded_path = None
for p in csv_paths:
    if os.path.exists(p):
        df = pd.read_csv(p)
        loaded_path = p
        break

if df is None:
    raise FileNotFoundError(f"Could not locate Microsoft Excel Security Classification dataset. Checked: {csv_paths}")

print(f"Loaded dataset from: {loaded_path}")

# Sanitize column headers: strip outer quotes, spaces, and non-alphanumeric punctuation
df.columns = [re.sub(r'[^\w\s]', '_', c).strip().replace(' ', '_') for c in df.columns]

print(f"Dataset Shape: {df.shape[0]:,} rows (records) and {df.shape[1]} columns (attributes)")
print("First 5 rows of the sanitized dataset:")
display(df.head(5))


In [ ]:
# 2.2 Data Integrity Check: Missing Values, Duplicates, and Statistical Profiles
print("=== 1. DATA TYPES & NON-NULL SUMMARY ===")
df.info()

print("\n=== 2. MISSING VALUES AUDIT ===")
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print(f"Zero missing values detected across all {df.shape[1]} attributes.")
else:
    print(null_counts[null_counts > 0])

print("\n=== 3. DUPLICATE RECORD AUDIT ===")
dup_count = df.duplicated().sum()
print(f"Total duplicate records: {dup_count:,}")

print("\n=== 4. SUMMARY STATISTICS OF NUMERICAL FEATURES ===")
display(df.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])


### 2.2 Selecting Features & Excluding Identifier
In Microsoft Excel Security Classification, each sample represents an examined document file.
- **Identifier:** `file_name` is excluded from model training to prevent identity memorization.
- **Target Variable:** `label` is the supervised ground truth binary indicator:
  - `0`: Benign Document
  - `1`: Malicious / Weaponized Exploit Document
- **Input Features:** All remaining attributes capture lexical, structural, and syntactical security characteristics.


In [ ]:
# Define feature subsets and roles
identifier_col = "file_name"
target_col = 'label'

input_features = [c for c in df.columns if c not in [identifier_col, target_col] and c is not None]

# Automatically generate structured feature metadata
metadata = []
if identifier_col and identifier_col in df.columns:
    metadata.append({'Attribute': identifier_col, 'Role': 'Excluded', 'Data Type': 'String', 'Description': 'Unique document filename identifier'})

for col in input_features:
    dtype_str = str(df[col].dtype)
    n_unique = df[col].nunique()
    if n_unique == 2:
        d_type = 'Binary Flag (0/1)'
    elif 'int' in dtype_str:
        d_type = 'Integer (Count / Length)'
    else:
        d_type = 'Continuous (Ratio / Entropy)'
    
    desc = f"Quantitative security attribute measuring {col.replace('_', ' ')}"
    metadata.append({'Attribute': col, 'Role': 'Input Feature', 'Data Type': d_type, 'Description': desc})

metadata.append({'Attribute': target_col, 'Role': 'Target Variable', 'Data Type': 'Binary Category (0/1)', 'Description': 'Excel Document Security Class (0 = Benign Spreadsheet, 1 = Malicious / Weaponized Macro Document)'})

metadata_df = pd.DataFrame(metadata)
print(f"Feature Selection Summary: {len(input_features)} input features selected, {1 if identifier_col else 0} identifier excluded, 1 target variable.")
display(metadata_df)


## 3. Visualizing and Understanding the Data
To develop deep data intuition for Microsoft Excel Security Classification, we conduct an in-depth exploratory data analysis (EDA):
1. **Univariate Distributions:** Mean vs. Median, variance, skewness, and extreme value outliers.
2. **Target Balance:** Supervised class distribution verification ($1:1$ prior ratio).
3. **Multivariate Correlation & Collinearity:** Inter-feature Pearson correlation ($r$) and condition indexing.
4. **Mutual Information (Information Gain):** Measuring non-linear dependence with target maliciousness ($I(X; Y)$).


In [ ]:
# 3.0 Comprehensive Univariate Distribution of Numerical Input Features
plot_features = input_features[:12] if len(input_features) > 12 else input_features
n_plots = len(plot_features)
ncols = 3
nrows = int(np.ceil(n_plots / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten() if n_plots > 1 else [axes]

for i, col in enumerate(plot_features):
    ax = axes[i]
    data = df[col].dropna()
    mean_val = data.mean()
    median_val = data.median()
    skew_val = data.skew()
    
    shape_label = 'Right-Skewed' if skew_val > 0.3 else ('Left-Skewed' if skew_val < -0.3 else 'Symmetric / Uniform')
    is_discrete = data.nunique() <= 10
    bins = min(data.nunique(), 30) if is_discrete else 30
    
    sns.histplot(data, bins=bins, kde=not is_discrete, color='#3498db', edgecolor='black', alpha=0.65, ax=ax)
    
    if abs(mean_val) > 10000:
        mean_str = f'{mean_val*1e-3:.1f}k'
        med_str = f'{median_val*1e-3:.1f}k'
    else:
        mean_str = f'{mean_val:.2f}'
        med_str = f'{median_val:.2f}'
        
    ax.axvline(mean_val, color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean: {mean_str}')
    ax.axvline(median_val, color='#2ecc71', linestyle='-', linewidth=2, label=f'Median: {med_str}')
    
    ax.set_title(f'{col}\n[{shape_label} | Skew: {skew_val:.2f}]', fontweight='bold', fontsize=10)
    ax.set_xlabel(col.replace('_', ' ').title(), fontweight='bold', fontsize=8.5)
    ax.set_ylabel('Count', fontweight='bold', fontsize=8.5)
    ax.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9, fontsize=8)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(f'Univariate Distributions of Key Numerical Features (N={len(df):,})', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# 3.1 Target Variable (label) Distribution Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

target_counts = df[target_col].value_counts().sort_index()
target_pcts = df[target_col].value_counts(normalize=True).sort_index() * 100
colors = ['#2ecc71', '#e74c3c']

# Count Barplot
bars = axes[0].bar(['Benign (0)', 'Malicious (1)'], target_counts.values, color=colors, edgecolor='black', width=0.55)
axes[0].set_title('Target Class Absolute Frequency Distribution', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Sample Count', fontweight='bold')
axes[0].set_ylim(0, max(target_counts.values) * 1.15)
for bar in bars:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + (max(target_counts.values)*0.02),
                 f'{int(yval):,} ({yval/len(df)*100:.1f}%)', ha='center', va='bottom', fontweight='bold')

# Percentage Donut Chart
axes[1].pie(target_pcts.values, labels=['Benign (0)', 'Malicious (1)'], autopct='%1.2f%%',
            startangle=90, colors=colors, explode=(0.04, 0),
            wedgeprops={'edgecolor': 'black', 'linewidth': 1.2, 'antialiased': True},
            textprops={'fontweight': 'bold', 'fontsize': 11})
centre_circle = plt.Circle((0,0), 0.65, fc='white', edgecolor='black', linewidth=1.0)
axes[1].add_artist(centre_circle)
axes[1].set_title('Target Class Proportions (Prior Balance)', fontweight='bold', fontsize=12)

plt.suptitle('Microsoft Excel Security Classification: Supervised Ground-Truth Class Balance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Key Findings on Target Variable:")
for idx, (cnt, pct) in enumerate(zip(target_counts.values, target_pcts.values)):
    print(f"  Class {idx} (['Benign (0)', 'Malicious (1)'][idx]): {cnt:,} samples ({pct:.2f}%)")
print(f"  Class Balance Ratio: {min(target_counts.values)/max(target_counts.values):.3f} (Ideal balance = 1.00)")


In [ ]:
# 3.2 Feature Discrimination: Class Separability for Top Attributes
sample_cols = input_features[:min(4, len(input_features))]
fig, axes = plt.subplots(1, len(sample_cols), figsize=(4.5 * len(sample_cols), 4))
if len(sample_cols) == 1:
    axes = [axes]

for i, col in enumerate(sample_cols):
    sns.boxplot(data=df, x=target_col, y=col, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'{col}\nby Security Class', fontweight='bold', fontsize=10)
    axes[i].set_xticklabels(['Benign (0)', 'Malicious (1)'])
    axes[i].set_xlabel('')

plt.suptitle('Class-Conditional Distribution Discrepancies across Discriminative Features', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# 3.3 Correlation Heatmap & Multicollinearity Analysis
corr_cols = input_features[:min(15, len(input_features))] + [target_col]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title(f'Pearson Correlation Matrix (Selected Attributes & Target)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Extract target correlations
target_corr = corr_matrix[target_col].drop(target_col).sort_values()
print("\n=== TOP LINEAR CORRELATES WITH MALICIOUS TARGET ===")
print("Strongest Negative Linear Drivers:")
print(target_corr.head(3))
print("\nStrongest Positive Linear Drivers:")
print(target_corr.tail(3))


In [ ]:
# 3.4 Mutual Information (Information Gain) Analysis
X_temp = df[input_features].fillna(0)
y_temp = df[target_col]

# Estimate Mutual Information scores
mi_scores = mutual_info_classif(X_temp, y_temp, discrete_features='auto', random_state=SEED)
mi_series = pd.Series(mi_scores, index=input_features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(5, int(len(mi_series[:15]) * 0.35))))
top_mi = mi_series.head(15).sort_values(ascending=True)
ax.barh(top_mi.index, top_mi.values, color='#9b59b6', edgecolor='black', alpha=0.85)
ax.set_title('Top Features Ranked by Mutual Information with Security Label (I(X; Y))', fontweight='bold', fontsize=12)
ax.set_xlabel('Mutual Information Score (Nats)', fontweight='bold')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

print("Mutual Information Ranking Summary (Top 10 Features):")
mi_df = pd.DataFrame({'Feature': top_mi.index[::-1], 'Mutual_Information': top_mi.values[::-1]})
display(mi_df.head(10))


---
### 3.5 Comprehensive Summary of Exploratory Data Analysis & Modeling Implications

| Dimension | Key Empirical Observation | Modeling & Optimization Implication |
| :--- | :--- | :--- |
| **Class Balance** | Dataset displays balanced distribution across Benign (0) and Malicious (1) records. | Accuracy, F1-Score, and ROC-AUC are all mathematically reliable; no synthetic SMOTE resampling required. |
| **Scale Disparity** | Feature values span ratios (0 to 1) to large integer counts (up to thousands/millions). | **Mandatory Feature Scaling:** Standardisation (`StandardScaler`, $z = (x - \mu)/\sigma$) fitted strictly on train data to prevent exploding gradients. |
| **Non-Linear Dynamics** | Attributes show high Mutual Information despite moderate linear Pearson correlation ($|r|$). | Justifies non-linear activation functions (ReLU) in Neural Networks and non-linear splitting boundaries in Tree Ensembles. |
| **Data Hygiene** | Zero missing values detected post-sanitization; uninformative identifiers safely excluded. | Eliminates data leakage and prevents spurious identifier memorization during model training. |
---


---
## 4. Data Preprocessing & Splitting Pipeline

To guarantee rigorous empirical evaluation and zero data leakage:
1. **Feature Matrix ($X$) & Target Vector ($y$):** Partition attributes into strictly separated feature matrices and target vectors.
2. **Stratified Splitting:** Partition dataset into $70\%$ Training split and $30\%$ Held-out Test split.
3. **Inner Validation Split:** Further partition the $70\%$ training split into $80\%$ Inner Training ($56\%$ of total) and $20\%$ Validation ($14\%$ of total).
4. **Single StandardScaler Pipeline:** Compute $(\mu, \sigma)$ strictly on inner training data; transform validation and test sets using training statistics.


In [ ]:
# ==============================================================================
# 4.1 Feature Matrix (X) and Target Vector (y) Construction
# ==============================================================================
X = df[input_features].copy()
y = df[target_col].copy().astype(int)
feature_cols = list(X.columns)

print("=== FEATURE & TARGET SPECIFICATION ===")
print(f"Total Records:           {len(df):,}")
print(f"Input Feature Matrix X:  {X.shape} ({len(feature_cols)} numerical attributes)")
print(f"Target Vector y:         {y.shape} (Binary class: 0=Benign, 1=Malicious)")
print(f"Class Proportions:       Benign: {(y==0).sum():,} ({(y==0).mean()*100:.2f}%) | Malicious: {(y==1).sum():,} ({(y==1).mean()*100:.2f}%)")


In [ ]:
# ==============================================================================
# 4.2 Stratified Partitioning: 70% Train / 30% Test + 20% Inner Validation
# ==============================================================================
# Step 1: Split entire dataset into 70% Train and 30% Held-Out Test
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

# Step 2: Split 70% Train into 80% Inner Train and 20% Validation
X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
    X_train_raw, y_train,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train
)

print("=== STRATIFIED DATASET PARTITIONING SUMMARY ===")
print(f"Total Dataset:           {len(df):,} records (100.0%)")
print(f"Training Split (70%):    {len(X_train_raw):,} records | Malicious Rate: {y_train.mean()*100:.2f}%")
print(f"  * Inner Train (80%):   {len(X_tr_raw):,} records | Malicious Rate: {y_tr.mean()*100:.2f}%")
print(f"  * Validation (20%):    {len(X_val_raw):,} records | Malicious Rate: {y_val.mean()*100:.2f}%")
print(f"Held-Out Test Set (30%): {len(X_test_raw):,} records | Malicious Rate: {y_test.mean()*100:.2f}%")


In [ ]:
# ==============================================================================
# 4.3 Feature Standardization (Single StandardScaler fitted on Inner Train)
# ==============================================================================
scaler = StandardScaler()

# Fit scaler strictly on inner training split
X_tr_arr = scaler.fit_transform(X_tr_raw)
X_val_arr = scaler.transform(X_val_raw)
X_test_arr = scaler.transform(X_test_raw)

# Convert targets to numpy arrays
y_tr_arr = y_tr.to_numpy(dtype=np.float32)
y_val_arr = y_val.to_numpy(dtype=np.float32)
y_test_arr = y_test.to_numpy(dtype=np.float32)

# Verify parameters
scaler_param_df = pd.DataFrame({
    'Feature': feature_cols,
    'Train Mean (mu)': np.round(scaler.mean_, 4),
    'Train Std (sigma)': np.round(scaler.scale_, 4)
})

print("=== STANDARDSCALER PARAMETERS (Fitted on Training Set Only) ===")
display(scaler_param_df.head(10))
print(f"\nInput Tensors Ready: Train={X_tr_arr.shape}, Val={X_val_arr.shape}, Test={X_test_arr.shape}")


---
## 7. Task #3: Hyperparameter Tuning

Hyperparameters determine the architectural capacity, optimization trajectory, and learning dynamics of the Multilayer Perceptron. Unlike weights $\mathbf{W}$ and biases $\mathbf{b}$, hyperparameters are **not directly learned through backpropagation**; they are chosen prior to training.

### Systematic Experimental Protocol (8 Distinct Configurations):
We systematically explore the hyperparameter space across five key axes:
1. **Architectural Depth (1 to 4 hidden layers):** `[16]`, `[32, 16]`, `[64, 32, 16]`, `[64, 32, 16, 8]`
2. **Layer Width (Neuron capacity):** `[32, 16]` vs `[64, 32]`
3. **Mini-Batch Size:** $B \in \{16, 32, 64\}$
4. **Learning Rate:** $\eta \in \{0.001, 0.01\}$
5. **Optimizer Family:** Adaptive (Adam) vs Inertial Momentum (SGD with Momentum)

*All models are evaluated on the identical 13-feature input space using 70/30 stratified splitting, single `StandardScaler` fitted on inner train data, for 500 epochs.*


In [ ]:
# ==============================================================================
# 7.1 Systematic Hyperparameter Tuning Pipeline (8 Controlled Configurations)
# ==============================================================================
import os
os.environ['KERAS_BACKEND'] = 'torch'
import time
import json
import numpy as np
import pandas as pd
import keras
from keras import layers, models, optimizers
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from IPython.display import display

RUN_HYPERPARAMETER_TUNING = True
EPOCHS = 500
SEED = 42

configurations = [
    {
        'config_id': 'C1',
        'desc': '1 Layer [16], Adam (lr=0.001), Batch=32',
        'layers': [16],
        'lr': 0.001,
        'batch_size': 32,
        'optimizer_type': 'adam'
    },
    {
        'config_id': 'C2',
        'desc': '2 Layers [32, 16] (Baseline), Adam (lr=0.001), Batch=32',
        'layers': [32, 16],
        'lr': 0.001,
        'batch_size': 32,
        'optimizer_type': 'adam'
    },
    {
        'config_id': 'C3',
        'desc': '2 Layers [64, 32], Adam (lr=0.001), Batch=32',
        'layers': [64, 32],
        'lr': 0.001,
        'batch_size': 32,
        'optimizer_type': 'adam'
    },
    {
        'config_id': 'C4',
        'desc': '3 Layers [64, 32, 16], Adam (lr=0.001), Batch=32',
        'layers': [64, 32, 16],
        'lr': 0.001,
        'batch_size': 32,
        'optimizer_type': 'adam'
    },
    {
        'config_id': 'C5',
        'desc': '4 Layers [64, 32, 16, 8], Adam (lr=0.001), Batch=32',
        'layers': [64, 32, 16, 8],
        'lr': 0.001,
        'batch_size': 32,
        'optimizer_type': 'adam'
    },
    {
        'config_id': 'C6',
        'desc': '2 Layers [32, 16], Adam (lr=0.001), Batch=16',
        'layers': [32, 16],
        'lr': 0.001,
        'batch_size': 16,
        'optimizer_type': 'adam'
    },
    {
        'config_id': 'C7',
        'desc': '2 Layers [32, 16], Adam (lr=0.001), Batch=64',
        'layers': [32, 16],
        'lr': 0.001,
        'batch_size': 64,
        'optimizer_type': 'adam'
    },
    {
        'config_id': 'C8',
        'desc': '2 Layers [32, 16], SGD+Momentum (lr=0.01, b=0.9), Batch=32',
        'layers': [32, 16],
        'lr': 0.01,
        'batch_size': 32,
        'optimizer_type': 'sgd_momentum'
    }
]

task3_results = []
task3_histories = {}

def build_mlp_from_config(cfg, input_dim=10):
    keras.utils.set_random_seed(SEED)
    model = models.Sequential(name=f"MLP_{cfg['config_id']}")
    model.add(layers.Input(shape=(input_dim,), name='input_features'))
    for idx, units in enumerate(cfg['layers']):
        model.add(layers.Dense(units, activation='relu', name=f"dense_{idx+1}_{units}"))
    model.add(layers.Dense(1, activation='sigmoid', name='output_sigmoid'))
    return model

print("=" * 80)
print(f" STARTING TASK #3 HYPERPARAMETER TUNING BENCHMARK ({len(configurations)} CONFIGURATIONS)")
print("=" * 80)

for cfg in configurations:
    cid = cfg['config_id']
    desc = cfg['desc']
    print(f"\n[+] Training Configuration {cid}: {desc} ...")
    
    model = build_mlp_from_config(cfg)
    if cfg['optimizer_type'] == 'adam':
        opt = optimizers.Adam(learning_rate=cfg['lr'])
    else:
        opt = optimizers.SGD(learning_rate=cfg['lr'], momentum=0.9, nesterov=False)
        
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    
    t0 = time.time()
    hist = model.fit(
        X_tr_arr, y_tr_arr,
        validation_data=(X_val_arr, y_val_arr),
        epochs=EPOCHS,
        batch_size=cfg['batch_size'],
        verbose=0
    )
    t_elapsed = time.time() - t0
    
    # Test Evaluation
    y_pred_probs = model.predict(X_test_arr, batch_size=64, verbose=0).flatten()
    y_pred_class = (y_pred_probs >= 0.5).astype(int)
    
    test_loss = float(model.evaluate(X_test_arr, y_test_arr, batch_size=64, verbose=0)[0])
    test_acc = float(accuracy_score(y_test_arr, y_pred_class) * 100)
    test_prec = float(precision_score(y_test_arr, y_pred_class))
    test_rec = float(recall_score(y_test_arr, y_pred_class))
    test_f1 = float(f1_score(y_test_arr, y_pred_class))
    test_auc = float(roc_auc_score(y_test_arr, y_pred_probs))
    
    best_val_epoch = int(np.argmin(hist.history['val_loss']) + 1)
    best_val_loss = float(min(hist.history['val_loss']))
    
    task3_histories[cid] = {
        'loss': [float(x) for x in hist.history['loss']],
        'val_loss': [float(x) for x in hist.history['val_loss']],
        'accuracy': [float(x) for x in hist.history['accuracy']],
        'val_accuracy': [float(x) for x in hist.history['val_accuracy']]
    }
    
    res = {
        'Config ID': cid,
        'Architecture': str(cfg['layers']),
        'Optimizer': cfg['optimizer_type'],
        'LR': cfg['lr'],
        'Batch Size': cfg['batch_size'],
        'Trainable Params': model.count_params(),
        'Best Val Loss': round(best_val_loss, 4),
        'Best Val Epoch': best_val_epoch,
        'Test Loss': round(test_loss, 4),
        'Test Accuracy (%)': round(test_acc, 2),
        'Test F1-Score': round(test_f1, 4),
        'Test ROC-AUC': round(test_auc, 4),
        'Training Time (s)': round(t_elapsed, 1)
    }
    task3_results.append(res)
    print(f"    Finished {cid} in {t_elapsed:.1f}s | Test Acc: {test_acc:.2f}% | Best Val Loss: {best_val_loss:.4f} (Ep {best_val_epoch})")

task3_summary_df = pd.DataFrame(task3_results).sort_values(by='Test Accuracy (%)', ascending=False)
print("\n" + "=" * 80)
print(" TASK #3: HYPERPARAMETER TUNING EMPIRICAL SUMMARY")
print("=" * 80)
display(task3_summary_df)


In [ ]:
# ==============================================================================
# 7.2 Visualizing Hyperparameter Tuning Dynamics (Depth, Batch Size, Optimizer)
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# (a) Network Depth Effect on Validation Loss
depth_configs = ['C1', 'C2', 'C4', 'C5']
depth_labels = {'C1': '1 Layer [16]', 'C2': '2 Layers [32, 16]', 'C4': '3 Layers [64, 32, 16]', 'C5': '4 Layers [64, 32, 16, 8]'}
depth_colors = {'C1': '#3498db', 'C2': '#2ecc71', 'C4': '#e67e22', 'C5': '#e74c3c'}

for cid in depth_configs:
    if cid in task3_histories:
        axes[0, 0].plot(range(1, EPOCHS + 1), task3_histories[cid]['val_loss'], label=depth_labels[cid], color=depth_colors[cid], lw=1.8)
axes[0, 0].set_title('Figure 1(a): Architectural Depth Impact on Validation Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# (b) Batch Size Effect on Validation Loss
bs_configs = ['C6', 'C2', 'C7']
bs_labels = {'C6': 'Batch Size = 16', 'C2': 'Batch Size = 32 (Baseline)', 'C7': 'Batch Size = 64'}
bs_colors = {'C6': '#9b59b6', 'C2': '#2ecc71', 'C7': '#34495e'}

for cid in bs_configs:
    if cid in task3_histories:
        axes[0, 1].plot(range(1, EPOCHS + 1), task3_histories[cid]['val_loss'], label=bs_labels[cid], color=bs_colors[cid], lw=1.8)
axes[0, 1].set_title('Figure 1(b): Batch Size Impact on Generalisation', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Validation Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# (c) Test Accuracy Across All 8 Configurations
c_ids = task3_summary_df['Config ID']
accs = task3_summary_df['Test Accuracy (%)']
axes[1, 0].bar(c_ids, accs, color='#3498db', edgecolor='black', width=0.55)
axes[1, 0].set_title('Figure 1(c): Test Accuracy (%) Comparison (8 Configurations)', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Test Accuracy (%)')
axes[1, 0].set_ylim(min(accs) - 3, 100)
for i, v in enumerate(accs):
    axes[1, 0].text(i, v + 0.3, f"{v:.2f}%", ha='center', fontweight='bold', fontsize=9)
axes[1, 0].grid(axis='y', alpha=0.3)

# (d) Training Time Trade-off
times = task3_summary_df['Training Time (s)']
axes[1, 1].bar(c_ids, times, color='#e67e22', edgecolor='black', width=0.55)
axes[1, 1].set_title('Figure 1(d): Computational Cost (Training Time in Seconds)', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Time (s)')
for i, v in enumerate(times):
    axes[1, 1].text(i, v + 2, f"{v:.1f}s", ha='center', fontweight='bold', fontsize=9)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.suptitle('Figure 1: Hyperparameter Tuning Multi-Axis Dynamics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### Task #3 Analytical Questions & Answers

#### Question 1: Which hyperparameters have the greatest effect on performance?
1. **Learning Rate & Optimizer Choice:** The optimizer and its step-size $\eta$ govern the gradient descent trajectory. A learning rate that is too aggressive causes divergence, while adaptive optimizers (Adam) achieve rapid initial convergence.
2. **Architectural Depth and Width:** Moving from a single hidden layer `[16]` (under-capacity) to two hidden layers `[32, 16]` provides sufficient expressive power to separate non-linear EXCEL script and link boundaries.
3. **Batch Size:** Dictates the variance of stochastic gradient estimates. Smaller batch sizes ($B=16$) introduce gradient noise that aids regularization but slows down execution, whereas larger batch sizes ($B=64$) provide faster hardware parallelization with smoother trajectories.

---

#### Question 2: Does increasing the number of hidden layers always improve prediction?
- **No.** Deeper architectures (`[64, 32, 16]` and `[64, 32, 16, 8]`) do not automatically yield higher test accuracy:
  - **Over-Parameterization:** With 10 input features, adding 4 dense layers inflates the number of trainable parameters. Without explicit regularization, deeper networks memorize training-set idiosyncrasies, causing the validation loss to diverge earlier.
  - **Diminishing Returns:** For tabular and engineered EXCEL features, a compact 2-hidden-layer network `[32, 16]` captures the non-linear interactions without unnecessary complexity.

---

#### Question 3: How does batch size affect training time and model performance?
- **Small Batch Size ($B = 16$):** Updates weights $\frac{N}{16}$ times per epoch. High gradient noise acts as implicit stochastic regularization, but memory transfers and non-vectorized overhead double the training time.
- **Large Batch Size ($B = 64$):** Maximizes vectorized GPU/CPU throughput, drastically reducing total wall-clock training time. However, very large batches can converge to sharper minima with slightly reduced test generalization.
- **Balanced ($B = 32$):** Balances computational efficiency and generalization.

---

#### Question 4: Why should hyperparameters not be selected using test-set performance?
- **Data Leakage & Optimism Bias:** If hyperparameters (depth, learning rate, batch size) are selected based on test-set metrics, information from the test set implicitly leaks into the modeling process.
- **Unbiased Generalization:** The test set must remain strictly held-out until the final model evaluation to provide an honest, unbiased assessment of real-world generalization on unseen webpages. All hyperparameter selections must be guided by cross-validation or a dedicated validation split.
